# OpenMontage — HARNESS Full Run (Kaggle T4 x2)

The **headless harness** path: one command turns an input into a finished,
QC-passed video — no IDE agent, no UI.

```bash
python -m harness run <input> --channel <id> [--motion auto|code2motion|template]
```

This notebook runs the whole stack on a free Kaggle T4 x2 session:

1. **Setup** — clone, deps, Node 22 (Remotion 4 + HyperFrames need it)
2. **Input** — write a `script.json` (or paste text / upload a voiceover —
   the harness accepts all three)
3. **Run** — `harness run`: Piper narration → whisper captions (GPU 0) →
   Open-Sora CLIP b-roll (GPU 1) → branded creative director → Remotion
   body → HyperFrames intro/outro (brand template or **code2motion**) →
   concat → QC tripwires → review
4. **Verify** — playback + QC artifacts

Optional: attach Kaggle secrets `LLM_BASE_URL` / `LLM_API_KEY` /
`LLM_MODEL` (or any provider key) to unlock LLM planning + code2motion.
No keys? Everything still runs — seeded director + brand templates.


## 0 · Config

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path("/kaggle/working/OpenMontage-beest")
WORK = Path("/kaggle/working/harness-run")
INPUT = WORK / "input"            # the harness input contract
OUT = WORK / "renders"

# Kaggle secrets (optional — LLM planning + code2motion + creative brief)
from kaggle_secrets import UserSecretsClient
try:
    sec = UserSecretsClient()
    for var, secret in [("LLM_BASE_URL", "LLM_BASE_URL"),
                        ("LLM_API_KEY", "LLM_API_KEY"),
                        ("LLM_MODEL", "LLM_MODEL")]:
        try:
            os.environ[var] = sec.get_secret(secret)
        except Exception:
            pass
except Exception:
    pass

CHANNEL = "tech-deep"             # try: default | tech-deep | story-loops
MOTION = "auto"                   # auto | code2motion | template

print("LLM configured:", bool(os.environ.get("LLM_API_KEY")))
print("GPUs:", subprocess.run(["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"],
                              capture_output=True, text=True).stdout.strip() or "none")

## 1 · Clone + install

Repo is public — plain clone, no token. GPU deps (torch, lancedb,
faster-whisper, transformers for CLIP) + core deps + Piper TTS. Node 22
tarball goes in front of PATH: Kaggle's stock Node is too old for
Remotion 4 / HyperFrames.

In [ ]:
t0 = time.time()

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/fafa433224/OpenMontage-beest.git", str(REPO)], check=True)
print("repo @", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

NODE_VER = "22.14.0"
node22 = Path(f"/usr/local/node-v{NODE_VER}")
if not node22.exists():
    tar = f"node-v{NODE_VER}-linux-x64.tar.xz"
    subprocess.run(f"curl -fsSL https://nodejs.org/dist/v{NODE_VER}/{tar} -o /tmp/{tar}", shell=True, check=True)
    subprocess.run(f"tar -xJf /tmp/{tar} -C /usr/local", shell=True, check=True)
os.environ["PATH"] = f"{node22}/bin:" + os.environ.get("PATH", "")
print("node:", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip())

r = subprocess.run(f"{sys.executable} -m pip install -q -r {REPO}/requirements.txt "
                   f"-r {REPO}/requirements-gpu.txt piper-tts 2>&1 | tail -5", shell=True, text=True)
print(r.stdout)
subprocess.run(f"{sys.executable} -m pip install -q 'av>=13.0,<14'", shell=True, check=True)

# Chrome shared libs for Remotion/HyperFrames headless shells
subprocess.run("apt-get install -y -q libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 "
               "libcups2 libdrm2 libxkbcommon0 libatspi2.0-0 libxcomposite1 libxdamage1 "
               "libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2 2>&1 | tail -2",
               shell=True, check=False)
print(f"setup done in {(time.time()-t0)/60:.1f} min")

## 2 · Harness doctor — verify everything before burning GPU hours

In [ ]:
r = subprocess.run(f"{sys.executable} -m harness doctor", shell=True, text=True, cwd=str(REPO))
print(r.stdout)
assert r.returncode in (0, 1), "doctor crashed"
# The whisper/llm/open-sora lines may be '!!' — the run still works with
# fallbacks; check them with your eyes before long runs.

## 3 · Input — any of these three

- **script.json** (used below): full control of sections
- a **text file**: the LLM plans hook → sections → CTA (or heuristic split)
- a **voiceover file**: voiceover-first — whisper derives the sections

Write your own topic here, or upload a voiceover to `/kaggle/working` and
pass its path to `harness run` instead.

In [ ]:
INPUT.mkdir(parents=True, exist_ok=True)

script = {
    "topic": "How black holes form",
    "title": "How Black Holes Form",
    "broll_moods": ["neon city rain", "macro water droplets", "star field timelapse"],
    "qa_notes": ["verify duration sanity", "verify word coverage above 90 percent"],
    "sections": [
        {"id": "s0", "heading": "A star runs out of fuel",
         "text": ("Every black hole begins as a massive star, at least twenty times heavier than our sun. "
                  "For millions of years, the star burns hydrogen into helium, and that outward pressure "
                  "holds the star up against its own crushing gravity.")},
        {"id": "s1", "heading": "Collapse in less than a second",
         "text": ("When the fuel finally runs out, the pressure vanishes and gravity wins instantly. "
                  "The core collapses in under a second, and the outer layers blast away in a supernova "
                  "explosion that can outshine a galaxy.")},
        {"id": "s2", "heading": "The point of no return",
         "text": ("What remains is packed into a region a few kilometers wide. Around it sits the event "
                  "horizon, a boundary where escape velocity exceeds the speed of light. Nothing that "
                  "crosses it can ever come back.")},
        {"id": "s3", "heading": "Why they matter",
         "text": ("Studying black holes pushes physics to its limits, testing Einstein in the most extreme "
                  "laboratory in the universe. Remember: every galaxy we have observed hosts a giant "
                  "black hole at its center.")},
    ],
}
(INPUT / "script.json").write_text(json.dumps(script, indent=2))
print("script.json written:", len(script["sections"]), "sections")

## 4 · (Optional) Open-Sora local corpus — real stock footage, zero keys

Skip this cell to use the procedural synth b-roll. Run it to give the
renderer a CLIP-searchable corpus of real CC0 clips (a ~100MB slice of
the LanguageBind/Open-Sora-Plan mixkit shard + LLaVA captions).

In [ ]:
OPEN_SORA_DIR = Path("/kaggle/temp/open-sora-dataset")
CLIPS_DIR = OPEN_SORA_DIR / "clips"

if not OPEN_SORA_DIR.exists():
    OPEN_SORA_DIR.mkdir(parents=True)
    subprocess.run(f"curl -fsSL https://huggingface.co/datasets/LanguageBind/Open-Sora-Plan/resolve/main/raw_data/mixkit/mixkit-100.tar.gz | tar -xz -C {CLIPS_DIR.mkdir(parents=True) or CLIPS_DIR}", shell=True, check=False)
    subprocess.run(f"curl -fsSL https://huggingface.co/datasets/LanguageBind/Open-Sora-Plan/resolve/main/raw_data/mixkit/mixkit-caption.jsonl -o {OPEN_SORA_DIR}/caption.jsonl", shell=True, check=False)

os.environ["OPEN_SORA_DIR"] = str(OPEN_SORA_DIR)
clips = list(CLIPS_DIR.glob("*.mp4")) if CLIPS_DIR.exists() else []
print("clips available:", len(clips))

if clips:
    r = subprocess.run(f"{sys.executable} tools/video/open_sora_index.py build", shell=True,
                       text=True, cwd=str(REPO), capture_output=True)
    print(r.stdout[-600:] or r.stderr[-600:])

## 5 · RUN — one command, the whole production

`harness run` = plan → narration (Piper) → whisper captions (GPU 0) →
Open-Sora b-roll (GPU 1) → branded creative director → Remotion body →
HyperFrames intro/outro → concat → QC → review.

If the session dies: `python -m harness resume <project-id>` continues
from the last checkpoint. GPU sessions are 30h/week — resume, don't restart.

In [ ]:
t0 = time.time()
cmd = [sys.executable, "-m", "harness", "run", str(INPUT / "script.json"),
       "--channel", CHANNEL, "--motion", MOTION,
       "--out", str(OUT), "--concurrency", "4"]
print("$", " ".join(cmd[2:]), flush=True)
r = subprocess.run(" ".join(cmd), shell=True, text=True, cwd=str(REPO))
print(f"\nharness exit={r.returncode} in {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "harness run failed — see log above"

## 6 · Verify — playback + QC artifacts

In [ ]:
from IPython.display import Video, display

finals = sorted(OUT.glob("final*.mp4")) if OUT.exists() else []
assert finals, "no final video rendered"
final = finals[-1]
print("FINAL:", final, f"{final.stat().st_size/2**20:.1f} MB")
display(Video(str(final), embed=True, width=800))

qc = OUT / "final-qc.json"
if qc.exists():
    print(qc.read_text())

timing = OUT / "final-timing.json"
if timing.exists():
    t = json.loads(timing.read_text())
    print("channel:", t.get("channel"), "| motion:", json.dumps(t.get("motion", {})))
    print("b-roll source:", t.get("broll_source", "synth (no Open-Sora index)"))

## What to try next

- **Channels**: `--channel story-loops` (calm cinematic) vs `tech-deep`
  (fast technical) — same script, two distinct branded outputs; write your
  own `channels/<id>/brand.yaml` (schema: `schemas/brands/channel.schema.json`).
- **Code2motion**: with an LLM secret attached, `--motion code2motion`
  makes the model author bespoke intro/outro choreography (falls back to
  the brand template automatically).
- **Voiceover input**: upload your own narration and run
  `harness run /kaggle/working/myvoiceover.mp3 --channel tech-deep` —
  the voiceover becomes the clock.
- **Resume**: kill the session mid-run on purpose, then
  `python -m harness resume <project-id>`.